### 0. Imports and Setup


In [2]:
import zipfile

with zipfile.ZipFile("./data/EdStats_CSV.zip", 'r') as zip:
    zip.extractall("./data")

In [9]:
import pandas as pd
import numpy as np
import sklearn
import seaborn
import scipy
import plotly
import statsmodels
from IPython.display import display

SEED = 42
np.random.seed(SEED)

In [ ]:
df_data = pd.read_csv("./data/EdStatsData.csv")

df_country_series = pd.read_csv("./data/EdStatsCountry-Series.csv")
df_country = pd.read_csv("./data/EdStatsCountry.csv")

df_footnote= pd.read_csv("./data/EdStatsFootNote.csv")
df_series = pd.read_csv("./data/EdStatsSeries.csv")


In [66]:
df_footnote["SeriesCode"].value_counts()

SeriesCode
SH.DYN.MORT             9226
SE.PRM.AGES             8771
SE.PRM.DURS             8771
SE.SEC.DURS             8619
SE.SEC.AGES             8581
                        ... 
LO.TIMSS.MAT4.HI.MA        1
LO.TIMSS.MAT4.INT.FE       1
LO.TIMSS.MAT4.INT.MA       1
LO.TIMSS.MAT4.LOW.FE       1
LO.TIMSS.MAT4.BL.FE        1
Name: count, Length: 1558, dtype: int64

In [58]:
# Kurze Regelungen:
# - Nur Nominal und Ordinale Skalen werden beibehalten. -> Beschreibungen, Redundante Namen, Lizensen werden verworfen.
# - Nur Zeitachse von 1999 bis 2026 wird behalten
# - 
display("df_data:", df_data.columns) # Zeitliche Angaben von 1970 bis 2100 + Spalte 69. -> Als Relevant definier ich 1999 bis 2026, Rest wird verworfen.

display("df_country_series:", df_country_series.columns)
display("df_country:",df_country.columns)

display("df_footnote",df_footnote.columns)
display("df_series:",df_series.columns)

'df_data:'

Index(['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code',
       '1970', '1971', '1972', '1973', '1974', '1975', '1976', '1977', '1978',
       '1979', '1980', '1981', '1982', '1983', '1984', '1985', '1986', '1987',
       '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995', '1996',
       '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005',
       '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014',
       '2015', '2016', '2017', '2020', '2025', '2030', '2035', '2040', '2045',
       '2050', '2055', '2060', '2065', '2070', '2075', '2080', '2085', '2090',
       '2095', '2100', 'Unnamed: 69'],
      dtype='str')

'df_country_series:'

Index(['CountryCode', 'SeriesCode', 'DESCRIPTION', 'Unnamed: 3'], dtype='str')

'df_country:'

Index(['Country Code', 'Short Name', 'Table Name', 'Long Name', '2-alpha code',
       'Currency Unit', 'Special Notes', 'Region', 'Income Group', 'WB-2 code',
       'National accounts base year', 'National accounts reference year',
       'SNA price valuation', 'Lending category', 'Other groups',
       'System of National Accounts', 'Alternative conversion factor',
       'PPP survey year', 'Balance of Payments Manual in use',
       'External debt Reporting status', 'System of trade',
       'Government Accounting concept', 'IMF data dissemination standard',
       'Latest population census', 'Latest household survey',
       'Source of most recent Income and expenditure data',
       'Vital registration complete', 'Latest agricultural census',
       'Latest industrial data', 'Latest trade data',
       'Latest water withdrawal data', 'Unnamed: 31'],
      dtype='str')

'df_footnote'

Index(['CountryCode', 'SeriesCode', 'Year', 'DESCRIPTION', 'Unnamed: 4'], dtype='str')

'df_series:'

Index(['Series Code', 'Topic', 'Indicator Name', 'Short definition',
       'Long definition', 'Unit of measure', 'Periodicity', 'Base Period',
       'Other notes', 'Aggregation method', 'Limitations and exceptions',
       'Notes from original source', 'General comments', 'Source',
       'Statistical concept and methodology', 'Development relevance',
       'Related source links', 'Other web links', 'Related indicators',
       'License Type', 'Unnamed: 20'],
      dtype='str')

In [51]:
display(df_country.columns)
display(df_country.sample(10))
display(df_country.info())
display(df_country.describe())

Index(['Country Code', 'Short Name', 'Table Name', 'Long Name', '2-alpha code',
       'Currency Unit', 'Special Notes', 'Region', 'Income Group', 'WB-2 code',
       'National accounts base year', 'National accounts reference year',
       'SNA price valuation', 'Lending category', 'Other groups',
       'System of National Accounts', 'Alternative conversion factor',
       'PPP survey year', 'Balance of Payments Manual in use',
       'External debt Reporting status', 'System of trade',
       'Government Accounting concept', 'IMF data dissemination standard',
       'Latest population census', 'Latest household survey',
       'Source of most recent Income and expenditure data',
       'Vital registration complete', 'Latest agricultural census',
       'Latest industrial data', 'Latest trade data',
       'Latest water withdrawal data', 'Unnamed: 31'],
      dtype='str')

,Country Code,Short Name,Table Name,Long Name,2-alpha code,Currency Unit,Special Notes,Region,Income Group,WB-2 code,...,IMF data dissemination standard,Latest population census,Latest household survey,Source of most recent Income and expenditure data,Vital registration complete,Latest agricultural census,Latest industrial data,Latest trade data,Latest water withdrawal data,Unnamed: 31
104,ITA,Italy,Italy,Italian Republic,IT,Euro,A simple multiplier is used to convert the nat...,Europe & Central Asia,High income: OECD,IT,...,Special Data Dissemination Standard (SDDS),2012,NaN,"Expenditure survey/budget survey (ES/BS), 2000",Yes,2010,2009.0,2012.0,2000,NaN
106,JOR,Jordan,Jordan,Hashemite Kingdom of Jordan,JO,Jordanian dinar,NaN,Middle East & North Africa,Upper middle income,JO,...,Special Data Dissemination Standard (SDDS),2004,"Demographic and Health Survey (DHS), 2012","Expenditure survey/budget survey (ES/BS), 2010",NaN,2007,2010.0,2012.0,2005,NaN
228,UZB,Uzbekistan,Uzbekistan,Republic of Uzbekistan,UZ,Uzbek sum,NaN,Europe & Central Asia,Lower middle income,UZ,...,NaN,1989,"Multiple Indicator Cluster Survey (MICS), 2006","Expenditure survey/budget survey (ES/BS), 2011",Yes,NaN,NaN,NaN,2005,NaN
109,KEN,Kenya,Kenya,Republic of Kenya,KE,Kenyan shilling,Fiscal year end: June 30; reporting period for...,Sub-Saharan Africa,Low income,KE,...,General Data Dissemination System (GDDS),2009,"Malaria Indicator Survey (MIS), 2010; HIV/Mate...","Integrated household survey (IHS), 2005/06",NaN,2009. Population and Housing Census.,2010.0,2010.0,2003,NaN
138,MDG,Madagascar,Madagascar,Republic of Madagascar,MG,Malagasy ariary,NaN,Sub-Saharan Africa,Low income,MG,...,General Data Dissemination System (GDDS),1993,"Malaria Indicator Survey (MIS), 2013","Priority survey (PS), 2010",NaN,NaN,2006.0,2012.0,2000,NaN
162,NIC,Nicaragua,Nicaragua,Republic of Nicaragua,NI,Nicaraguan gold cordoba,April 2013 database update: Based on official ...,Latin America & Caribbean,Lower middle income,NI,...,General Data Dissemination System (GDDS),2005,"Reproductive Health Survey (RHS), 2006/2007",Living Standards Measurement Study Survey (LSM...,NaN,2011,NaN,2012.0,2001,NaN
203,SVK,Slovak Republic,Slovak Republic,Slovak Republic,SK,Euro,A simple multiplier is used to convert the nat...,Europe & Central Asia,High income: OECD,SK,...,Special Data Dissemination Standard (SDDS),2011,"World Health Survey (WHS), 2003","Income survey (IS), 2009",Yes,2010,2009.0,2012.0,2007,NaN
110,KGZ,Kyrgyz Republic,Kyrgyz Republic,Kyrgyz Republic,KG,Kyrgyz som,NaN,Europe & Central Asia,Lower middle income,KG,...,Special Data Dissemination Standard (SDDS),2009,"Demographic and Health Survey (DHS), 2012","Expenditure survey/budget survey (ES/BS), 2012",Yes,2014,2010.0,2012.0,2006,NaN
223,UGA,Uganda,Uganda,Republic of Uganda,UG,Ugandan shilling,Fiscal year end: June 30; reporting period for...,Sub-Saharan Africa,Low income,UG,...,General Data Dissemination System (GDDS),2002,"HIV/AIDS Indicator Survey (AIS), 2011; Demogra...","Priority survey (PS), 2009/10",NaN,2008/ 2009,2000.0,2012.0,2002,NaN
161,NGA,Nigeria,Nigeria,Federal Republic of Nigeria,NG,Nigerian naira,Based on official government statistics releas...,Sub-Saharan Africa,Lower middle income,NG,...,General Data Dissemination System (GDDS),2006,"Demographic and Health Survey (DHS), 2013","Integrated household survey (IHS), 2010",NaN,2013,NaN,2012.0,2005,NaN


<class 'pandas.DataFrame'>
RangeIndex: 241 entries, 0 to 240
Data columns (total 32 columns):
 #   Column                                             Non-Null Count  Dtype  
---  ------                                             --------------  -----  
 0   Country Code                                       241 non-null    str    
 1   Short Name                                         241 non-null    str    
 2   Table Name                                         241 non-null    str    
 3   Long Name                                          241 non-null    str    
 4   2-alpha code                                       238 non-null    str    
 5   Currency Unit                                      215 non-null    str    
 6   Special Notes                                      145 non-null    str    
 7   Region                                             214 non-null    str    
 8   Income Group                                       214 non-null    str    
 9   WB-2 code            

None

,National accounts reference year,Latest industrial data,Latest trade data,Unnamed: 31
count,32.00000,107.000000,185.000000,0.0
mean,2001.53125,2008.102804,2010.994595,NaN
std,5.24856,2.616834,2.569675,NaN
min,1987.00000,2000.000000,1995.000000,NaN
25%,1996.75000,2007.500000,2011.000000,NaN
50%,2002.00000,2009.000000,2012.000000,NaN
75%,2005.00000,2010.000000,2012.000000,NaN
max,2012.00000,2010.000000,2012.000000,NaN


In [38]:
display(df_series.columns)
display(df_series.sample(10))
display(df_series.info())

# Fazit bisher: Alles unnötig bis Series Code und eventuell Topic

Index(['Series Code', 'Topic', 'Indicator Name', 'Short definition',
       'Long definition', 'Unit of measure', 'Periodicity', 'Base Period',
       'Other notes', 'Aggregation method', 'Limitations and exceptions',
       'Notes from original source', 'General comments', 'Source',
       'Statistical concept and methodology', 'Development relevance',
       'Related source links', 'Other web links', 'Related indicators',
       'License Type', 'Unnamed: 20'],
      dtype='str')

,Series Code,Topic,Indicator Name,Short definition,Long definition,Unit of measure,Periodicity,Base Period,Other notes,Aggregation method,...,Notes from original source,General comments,Source,Statistical concept and methodology,Development relevance,Related source links,Other web links,Related indicators,License Type,Unnamed: 20
2100,SABER.SCH.ATNM.GOAL3.LVL1,School Autonomy and Accountability (SABER),SABER: (School Autonomy and Accountability) Po...,Data Interpretation: 1=Latent; 2=Emerging; 3=E...,Data Interpretation: 1=Latent; 2=Emerging; 3=E...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,Systems Approach for Better Education Results ...,NaN,NaN,http://saber.worldbank.org/index.cfm,NaN,NaN,NaN,NaN
3470,UIS.SAP.23.GPV.G1,Population,Population of the official entrance age to sec...,NaN,Population of the age-group theoretically corr...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2090,SABER.SCH.ATNM.GOAL1.LVL1,School Autonomy and Accountability (SABER),SABER: (School Autonomy and Accountability) Po...,Data Interpretation: 1=Latent; 2=Emerging; 3=E...,Data Interpretation: 1=Latent; 2=Emerging; 3=E...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,Systems Approach for Better Education Results ...,NaN,NaN,http://saber.worldbank.org/index.cfm,NaN,NaN,NaN,NaN
1558,LO.TIMSS.MAT4,Learning Outcomes,TIMSS: Mean performance on the mathematics sca...,TIMSS: Mean performance on the mathematics sca...,TIMSS: Mean performance on the mathematics sca...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,International Association for the Evaluation o...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3043,UIS.GTVP.4.GPV.F,Post-Secondary/Non-Tertiary,Percentage of female students in post-secondar...,NaN,Total number of female students enrolled in ge...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2447,SP.POP.1014.FE.UN,Population,"Population, ages 10-14, female","Population, ages 10-14, female is the total nu...","Population, ages 10-14, female is the total nu...",NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,UNESCO Institute for Statistics (Derived),NaN,NaN,NaN,NaN,NaN,NaN,NaN
3196,UIS.NART.2.Q5,Education Equality,"UIS: Total net attendance rate, lower secondar...",Total number of students of the official lower...,Total number of students of the official lower...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3393,UIS.REPR.2.GPV.G1.M,Secondary,Repetition rate in Grade 1 of lower secondary ...,NaN,Number of male repeaters in a given grade in a...,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,UNESCO Institute for Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN
141,BAR.PRM.SCHL.5559.FE,Attainment,"Barro-Lee: Average years of primary schooling,...","Average years of primary schooling, 55-59, fem...","Average years of primary schooling, 55-59, fem...",NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,Robert J. Barro and Jong-Wha Lee: http://www.b...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
688,LO.EGRA.CWPM.SOM.2GRD,Learning Outcomes,EGRA: Correct Isolated Words Read Per Minute (...,Average total number of familiar words correct...,Average total number of familiar words correct...,NaN,NaN,NaN,EGRA,NaN,...,NaN,NaN,Early Grade Reading Assessment (EGRA): https:/...,NaN,NaN,NaN,NaN,NaN,NaN,NaN


<class 'pandas.DataFrame'>
RangeIndex: 3665 entries, 0 to 3664
Data columns (total 21 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   Series Code                          3665 non-null   str    
 1   Topic                                3665 non-null   str    
 2   Indicator Name                       3665 non-null   str    
 3   Short definition                     2156 non-null   str    
 4   Long definition                      3665 non-null   str    
 5   Unit of measure                      0 non-null      float64
 6   Periodicity                          99 non-null     str    
 7   Base Period                          314 non-null    str    
 8   Other notes                          552 non-null    str    
 9   Aggregation method                   47 non-null     str    
 10  Limitations and exceptions           14 non-null     str    
 11  Notes from original source           0 no

None

In [21]:
display(df_country_series.columns)
display(df_country_series.sample(10))
display(df_country_series.info())

# Fazit bisher:

Index(['CountryCode', 'SeriesCode', 'DESCRIPTION', 'Unnamed: 3'], dtype='str')

,CountryCode,SeriesCode,DESCRIPTION,Unnamed: 3
114,CUB,SP.POP.GROW,Data sources: United Nations World Population ...,NaN
243,IND,SP.POP.GROW,Data sources: United Nations World Population ...,NaN
344,MDA,NY.GDP.PCAP.PP.KD,Excludes Transnistria.,NaN
263,KAZ,SP.POP.GROW,Data sources: Statistical Agency of Kazakhstan...,NaN
59,BIH,SP.POP.TOTL,Data sources : United Nations World Population...,NaN
349,MDV,SP.POP.GROW,"Data sources: For 1960-1989, United Nations Wo...",NaN
119,CYP,NY.GDP.PCAP.PP.KD,Data are for areas under the effective control...,NaN
591,VNM,SP.POP.GROW,Data sources: General Statistics Office of Vie...,NaN
185,GEO,SP.POP.1564.TO.ZS,Including Abkhazia and South Ossetia.,NaN
20,ARG,NY.GDP.PCAP.PP.CD,Estimates are based on regression.,NaN


<class 'pandas.DataFrame'>
RangeIndex: 613 entries, 0 to 612
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   CountryCode  613 non-null    str    
 1   SeriesCode   613 non-null    str    
 2   DESCRIPTION  613 non-null    str    
 3   Unnamed: 3   0 non-null      float64
dtypes: float64(1), str(3)
memory usage: 19.3 KB


None

In [25]:
country = df_country_series[df_country_series["CountryCode"] == "DEU"]
country

,CountryCode,SeriesCode,DESCRIPTION,Unnamed: 3
132,DEU,SP.POP.GROW,Data sources : Eurostat,NaN
133,DEU,SP.POP.TOTL,Data sources : Eurostat,NaN


In [30]:
df_series_code = df_series[df_series["Series Code"].isin(["SP.POP.GROW" , "SP.POP.TOTL"])]
df_series_code

,Series Code,Topic,Indicator Name,Short definition,Long definition,Unit of measure,Periodicity,Base Period,Other notes,Aggregation method,...,Notes from original source,General comments,Source,Statistical concept and methodology,Development relevance,Related source links,Other web links,Related indicators,License Type,Unnamed: 20
2589,SP.POP.GROW,Health: Population: Dynamics,Population growth (annual %),Annual population growth rate. Population is b...,Annual population growth rate for year t is th...,NaN,Annual,NaN,NaN,Weighted average,...,NaN,NaN,Derived from total population. Population sour...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2590,SP.POP.TOTL,Health: Population: Structure,"Population, total",NaN,Total population is based on the de facto defi...,NaN,Annual,NaN,NaN,Sum,...,NaN,Relevance to gender indicator: disaggregating ...,(1) United Nations Population Division. World ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
display(df_country_series["CountryCode"].value_counts(), sort=True)
# CHI und CHN könnte gleich sein
display(df_country_series["SeriesCode"].value_counts(), sort=True) # 


display(df_series["Series Code"].value_counts(), sort=True)


CountryCode
GEO    18
MDA    18
CYP    12
MAR    12
MUS    12
       ..
WSM     2
YEM     2
ZAF     2
ZMB     2
ZWE     2
Name: count, Length: 211, dtype: int64

SeriesCode
SP.POP.TOTL          211
SP.POP.GROW          211
NY.GDP.PCAP.PP.CD     19
NY.GDP.PCAP.PP.KD     19
NY.GNP.PCAP.PP.CD     19
NY.GDP.MKTP.PP.KD     14
NY.GNP.MKTP.PP.CD     14
NY.GDP.MKTP.PP.CD     14
SP.POP.1564.TO.ZS     13
SP.POP.TOTL.MA.ZS     13
SP.POP.TOTL.FE.ZS     13
SP.POP.0014.TO.ZS     13
NY.GNP.PCAP.CD         6
NY.GDP.PCAP.CD         5
NY.GDP.PCAP.KD         5
SP.POP.1564.MA.IN      4
SP.POP.0014.TO         4
SP.POP.1564.TO         4
SP.POP.1564.FE.IN      4
SP.POP.0014.MA.IN      4
SP.POP.0014.FE.IN      4
Name: count, dtype: int64

Series Code
BAR.NOED.1519.FE.ZS            1
BAR.NOED.1519.ZS               1
BAR.NOED.15UP.FE.ZS            1
BAR.NOED.15UP.ZS               1
BAR.NOED.2024.FE.ZS            1
                              ..
UIS.XUNIT.USCONST.3.FSGOV      1
UIS.XUNIT.USCONST.4.FSGOV      1
UIS.XUNIT.USCONST.56.FSGOV     1
XGDP.23.FSGOV.FDINSTADM.FFD    1
XGDP.56.FSGOV.FDINSTADM.FFD    1
Name: count, Length: 3665, dtype: int64

In [47]:
# CHI wird fuer Chile stehen und CHN fuer China. CN exestiert nicht.

country_chi = df_country_series[df_country_series["CountryCode"] == "CHI"]
display(country_chi)

country_chn = df_country_series[df_country_series["CountryCode"] == "CHL"]
display(country_chn)

,CountryCode,SeriesCode,DESCRIPTION,Unnamed: 3
84,CHI,SP.POP.TOTL,Data sources : United Nations World Population...,NaN
85,CHI,SP.POP.0014.TO.ZS,"Refers to Guernsey, and Jersey.",NaN
86,CHI,SP.POP.1564.TO.ZS,"Refers to Guernsey, and Jersey.",NaN
87,CHI,SP.POP.GROW,Data sources: United Nations World Population ...,NaN
88,CHI,SP.POP.TOTL.FE.ZS,"Refers to Guernsey, and Jersey.",NaN
89,CHI,SP.POP.TOTL.MA.ZS,"Refers to Guernsey, and Jersey.",NaN


,CountryCode,SeriesCode,DESCRIPTION,Unnamed: 3
90,CHL,SP.POP.GROW,Data sources: United Nations World Population ...,NaN
91,CHL,SP.POP.TOTL,Data sources : United Nations World Population...,NaN


In [4]:
df.isnull().sum()

Country Name           0
Country Code           0
Indicator Name         0
Indicator Code         0
1970              814642
                   ...  
2085              835494
2090              835494
2095              835494
2100              835494
Unnamed: 69       886930
Length: 70, dtype: int64

In [5]:
df.tail(20)

,Country Name,Country Code,Indicator Name,Indicator Code,1970,1971,1972,1973,1974,1975,...,2060,2065,2070,2075,2080,2085,2090,2095,2100,Unnamed: 69
886910,Zimbabwe,ZWE,Wittgenstein Projection: Population in thousan...,PRJ.POP.ALL.NED.FE,NaN,NaN,NaN,NaN,NaN,NaN,...,4.13,2.58,1.71,1.19,0.86,0.66,0.56,0.51,0.49,NaN
886911,Zimbabwe,ZWE,Wittgenstein Projection: Population in thousan...,PRJ.POP.ALL.NED.MA,NaN,NaN,NaN,NaN,NaN,NaN,...,1.56,1.10,0.81,0.64,0.53,0.47,0.45,0.43,0.43,NaN
886912,Zimbabwe,ZWE,Wittgenstein Projection: Population in thousan...,PRJ.POP.ALL.NED.MF,NaN,NaN,NaN,NaN,NaN,NaN,...,5.69,3.68,2.53,1.82,1.39,1.14,1.01,0.94,0.91,NaN
886913,Zimbabwe,ZWE,Wittgenstein Projection: Population in thousan...,PRJ.POP.ALL.4.FE,NaN,NaN,NaN,NaN,NaN,NaN,...,576.34,636.47,699.49,764.63,832.26,903.02,978.19,1057.39,1139.21,NaN
886914,Zimbabwe,ZWE,Wittgenstein Projection: Population in thousan...,PRJ.POP.ALL.4.MA,NaN,NaN,NaN,NaN,NaN,NaN,...,590.14,629.83,674.80,724.51,779.53,840.53,908.72,983.44,1063.26,NaN
886915,Zimbabwe,ZWE,Wittgenstein Projection: Population in thousan...,PRJ.POP.ALL.4.MF,NaN,NaN,NaN,NaN,NaN,NaN,...,1166.47,1266.30,1374.29,1489.14,1611.78,1743.55,1886.91,2040.83,2202.47,NaN
886916,Zimbabwe,ZWE,Wittgenstein Projection: Population in thousan...,PRJ.POP.ALL.1.FE,NaN,NaN,NaN,NaN,NaN,NaN,...,282.09,241.06,201.98,167.68,137.86,113.78,93.18,77.30,65.11,NaN
886917,Zimbabwe,ZWE,Wittgenstein Projection: Population in thousan...,PRJ.POP.ALL.1.MA,NaN,NaN,NaN,NaN,NaN,NaN,...,212.14,179.40,150.11,126.43,106.82,91.70,78.21,67.56,59.06,NaN
886918,Zimbabwe,ZWE,Wittgenstein Projection: Population in thousan...,PRJ.POP.ALL.1.MF,NaN,NaN,NaN,NaN,NaN,NaN,...,494.23,420.47,352.09,294.10,244.68,205.47,171.39,144.86,124.17,NaN
886919,Zimbabwe,ZWE,Wittgenstein Projection: Population in thousan...,PRJ.POP.ALL.3.FE,NaN,NaN,NaN,NaN,NaN,NaN,...,3469.66,3493.32,3491.72,3468.64,3429.54,3378.61,3317.67,3248.91,3175.64,NaN


### 1. EDA


### 2. Feature Engineering


### 3. Modeling and Training


### 4. Evaluation
